In [59]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증'
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 차원 축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
from sklearn.cluster import estimate_bandwidth

# 학습 모델 저장을 위한 라이브러리
import pickle

In [60]:
# 데이터를 읽어온다.
df1 = pd.read_parquet(r'data/1.회원정보(train).parquet')
df2 = pd.read_parquet(r'data/신용정보_VIF용5.parquet')
df3 = pd.read_parquet(r'data/승인매출정보_변수추출_1차.parquet')
df4 = pd.read_parquet(r'data/4.청구입금정보(train).parquet')
df5 = pd.read_parquet(r'data/5.잔액정보(train).parquet')
df6 = pd.read_parquet(r'data/6.채널정보_VIF용2.parquet')

In [61]:
# 1. 먼저 기준이 되는 df1부터 시작
merged_df = df1.copy()

# 2. ID & 기준년월 기준으로 순차 병합
merged_df = merged_df.merge(df2, on=['ID', '기준년월'], how='inner')
merged_df = merged_df.merge(df3, on=['ID', '기준년월'], how='inner')
merged_df = merged_df.merge(df4, on=['ID', '기준년월'], how='inner')
merged_df = merged_df.merge(df5, on=['ID', '기준년월'], how='inner')
merged_df = merged_df.merge(df6, on=['ID', '기준년월'], how='inner')

In [62]:
test_df4 = pd.read_csv(r'test_merged.csv')

In [69]:
drop_cols = [
    'ID', '기준년월', '이용건수_신용_B0M', '이용건수_신용_R3M', '이용건수_신용_R6M',
    '이용건수_신판_B0M', '이용건수_신판_R3M', '이용건수_신판_R6M', '이용건수_신판_R12M',
    '이용건수_오프라인_B0M', '이용건수_오프라인_R3M', '이용건수_오프라인_R6M',
    '이용건수_일시불_B0M', '이용건수_일시불_R3M', '이용건수_일시불_R6M', '이용건수_일시불_R12M',
    '이용금액_R3M_신용', '이용금액_일시불_B0M', '이용금액_일시불_R3M', '이용금액_일시불_R6M', '이용금액_일시불_R12M',
    '이용금액_오프라인_B0M', '이용금액_오프라인_R3M', '이용금액_오프라인_R6M',
    '정상청구원금_B0M', '정상청구원금_B2M', '정상입금원금_B0M', '정상입금원금_B2M', '정상입금원금_B5M',
    '월중평잔_일시불', '월중평잔_일시불_B0M', '잔액_일시불_B1M', '잔액_일시불_B2M',
    '청구금액_B0', '청구금액_R3M', '_2순위신용체크구분',
    '_1순위카드이용건수', '_2순위카드이용건수', '_3순위쇼핑업종_이용금액',
    '교통_주유이용금액', '수신거부여부_DM', '수신거부여부_메일',
    '이용가맹점수', '보유여부_해외겸용_본인','_1순위카드이용금액', '이용금액_페이_온라인_B0M',
    '쇼핑_도소매_이용금액','쇼핑_마트_이용금액', '연체입금원금_B2M', '연체입금원금_B5M', '이용금액_온라인_B0M',
    '_2순위쇼핑업종_이용금액','_3순위업종_이용금액', '청구금액_R6M',
    '_2순위업종_이용금액','평잔_일시불_3M', '_1순위업종_이용금액'
]

train_df5 = merged_df.drop(columns=drop_cols)

In [70]:
drop_cols2 = [
    'ID', '기준년월', '이용건수_신용_B0M', '이용건수_신용_R3M', '이용건수_신용_R6M',
    '이용건수_신판_B0M', '이용건수_신판_R3M', '이용건수_신판_R6M', '이용건수_신판_R12M',
    '이용건수_오프라인_B0M', '이용건수_오프라인_R3M', '이용건수_오프라인_R6M',
    '이용건수_일시불_B0M', '이용건수_일시불_R3M', '이용건수_일시불_R6M', '이용건수_일시불_R12M',
    '이용금액_R3M_신용', '이용금액_일시불_B0M', '이용금액_일시불_R3M', '이용금액_일시불_R6M', '이용금액_일시불_R12M',
    '이용금액_오프라인_B0M', '이용금액_오프라인_R3M', '이용금액_오프라인_R6M',
    '정상청구원금_B0M', '정상청구원금_B2M', '정상입금원금_B0M', '정상입금원금_B2M', '정상입금원금_B5M',
    '월중평잔_일시불', '월중평잔_일시불_B0M', '잔액_일시불_B1M', '잔액_일시불_B2M',
    '청구금액_B0', '청구금액_R3M',
    '_1순위카드이용건수', '_2순위카드이용건수', '_3순위쇼핑업종_이용금액',
    '교통_주유이용금액', '수신거부여부_DM', '수신거부여부_메일', '이용카드수_신용',
    '이용가맹점수', '보유여부_해외겸용_본인','_1순위카드이용금액', '이용금액_페이_온라인_B0M',
    '쇼핑_도소매_이용금액','쇼핑_마트_이용금액', '연체입금원금_B2M', '연체입금원금_B5M', '이용금액_온라인_B0M',
    '_2순위쇼핑업종_이용금액','_3순위업종_이용금액', '청구금액_R6M',
    '_2순위업종_이용금액','평잔_일시불_3M', '_1순위업종_이용금액']
test_df5=test_df4.drop(columns=drop_cols2)

In [71]:
train_df5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400000 entries, 0 to 2399999
Data columns (total 21 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   소지카드수_이용가능_신용       int64 
 1   소지카드수_유효_신용         int64 
 2   이용가능여부_해외겸용_본인      int64 
 3   수신거부여부_TM           int64 
 4   이용금액_R3M_신용체크       int64 
 5   이용카드수_신용체크          int64 
 6   _2순위카드이용금액          int64 
 7   상향가능한도금액            int64 
 8   상향가능CA한도금액          int64 
 9   정상청구원금_B5M          int64 
 10  이용건수_신용_R12M        int64 
 11  최대이용금액_일시불_R12M     int64 
 12  _1순위교통업종_이용금액       int64 
 13  연체입금원금_B0M          int64 
 14  쇼핑_슈퍼마켓_이용금액        int64 
 15  _1순위쇼핑업종_이용금액       int64 
 16  연속유실적개월수_기본_24M_카드  int64 
 17  이용금액대               object
 18  할인건수_R3M            object
 19  잔액_일시불_B0M          int64 
 20  인입횟수_ARS_R6M        object
dtypes: int64(18), object(3)
memory usage: 384.5+ MB


In [72]:
test_df5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600000 entries, 0 to 599999
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   소지카드수_이용가능_신용       600000 non-null  int64 
 1   소지카드수_유효_신용         600000 non-null  int64 
 2   이용가능여부_해외겸용_본인      600000 non-null  int64 
 3   수신거부여부_TM           600000 non-null  int64 
 4   이용금액_R3M_신용체크       600000 non-null  int64 
 5   이용카드수_신용체크          600000 non-null  int64 
 6   _2순위카드이용금액          600000 non-null  int64 
 7   상향가능한도금액            600000 non-null  int64 
 8   상향가능CA한도금액          600000 non-null  int64 
 9   정상청구원금_B5M          600000 non-null  int64 
 10  이용건수_신용_R12M        600000 non-null  int64 
 11  최대이용금액_일시불_R12M     600000 non-null  int64 
 12  _1순위교통업종_이용금액       600000 non-null  int64 
 13  연체입금원금_B0M          600000 non-null  int64 
 14  쇼핑_슈퍼마켓_이용금액        600000 non-null  int64 
 15  _1순위쇼핑업종_이용금액       600000 non-null  int64 
 16  연속

In [73]:
train_df5.to_csv('train_VIF.csv', index=False, encoding='utf-8-sig')

In [74]:
test_df5.to_csv('test_VIF.csv', index=False, encoding='utf-8-sig')